# Домашнее задание 2. Компьютерный практикум 3.

Печенин Данила Михайлович. БПМ233.

In [1]:
import numpy as np
from numba import njit, prange, objmode
import matplotlib.pyplot as plt
import time

Note / Примечание:   
Besides installation of these libraries better to install scipy, as numba seems to be needed this but don't automatically download it (at least on my laptop).  
Помимо установки этих библиотек рекомендуется установить scipy, так как numba, похоже, нуждается в нём, но не скачивает его автоматически (по крайней мере, на моём ноутбуке).

In [2]:
@njit
def precompute_neighbors(Lx: int, Ly: int):
    """
    Precompute indices of right and bottom neighbors for all grid nodes.
    
    For each node (i, j) in the Lx × Ly grid, this function computes the indices
    of its right and bottom neighbors taking into account periodic boundary conditions.
    
    Parameters
    ----------
    Lx : int
        Number of nodes in the x-direction (horizontal size of the grid)
    Ly : int
        Number of nodes in the y-direction (vertical size of the grid)
    
    Returns
    -------
    right_neighbors : np.ndarray
        Array of shape (Lx, Ly, 2) containing indices of right neighbors.
        right_neighbors[i, j] = [i_right, j_right] where:
        - i_right = (i + 1) % Lx (periodic boundary in x-direction)
        - j_right = j (same row)
    bottom_neighbors : np.ndarray
        Array of shape (Lx, Ly, 2) containing indices of bottom neighbors.
        bottom_neighbors[i, j] = [i_bottom, j_bottom] where:
        - i_bottom = i (same column)
        - j_bottom = (j + 1) % Ly (periodic boundary in y-direction)
    """
    right_neighbors = np.zeros((Lx, Ly, 2), dtype=np.int32)
    bottom_neighbors = np.zeros((Lx, Ly, 2), dtype=np.int32)
    
    for i in range(Lx):
        for j in range(Ly):
            right_neighbors[i, j, 0] = (i + 1) % Lx
            right_neighbors[i, j, 1] = j
            
            bottom_neighbors[i, j, 0] = i
            bottom_neighbors[i, j, 1] = (j + 1) % Ly
    
    return right_neighbors, bottom_neighbors

In [3]:
@njit(parallel=True)
def compute_normed_mean_energy(Lx_array: np.ndarray, Ly: int, kT_array: np.ndarray, J: float = 1.0) -> np.ndarray:
    """
    Compute the normalized mean energy per site for the 2D Ising model. This function provides exact solution.
    
    This function calculates the average energy per spin site for a 2D Ising model
    with periodic boundary conditions using exact enumeration of all possible
    spin configurations. The computation exploits the energy symmetry E(σ) = E(-σ)
    to reduce the state space by half and uses parallel processing for efficiency.
    
    The Ising model Hamiltonian with nearest-neighbor interactions is:
        H = -J * Σ_{<i,j>} σ_i σ_j
    where σ_i ∈ {-1, +1} are spin variables and the sum is over nearest neighbors.
    
    Parameters
    ----------
    Lx_array : np.ndarray
        1D array of integer values representing the horizontal sizes of the gird.
        Each element specifies the number of spins along the x-direction.
    Ly : int
        Integer value representing the vertical size of the gird (fixed for all computations).
        Specifies the number of spins along the y-direction.
    kT_array : np.ndarray
        1D array of float values representing temperature values multiplied by 
        Boltzmann's constant (k*T).
    J : float, optional
        Coupling constant between neighboring spins. Default is 1.0 (ferromagnetic coupling).
    
    Returns
    -------
    normed_mean_energies : np.ndarray
        2D array of shape (len(Lx_array), len(kT_array)) containing the normalized
        mean energy per site ⟨E⟩/(Lx*Ly) for each combination of gird size and temperature.
        The array is organized such that normed_mean_energies[i, j] corresponds to
        Lx_array[i] and kT_array[j].
    
    See Also
    --------
    precompute_neighbors : Function that precomputes neighbor indices for efficiency.
    """
    Lx_array_len, kT_array_len = Lx_array.shape[0], kT_array.shape[0]
    normed_mean_energies = np.zeros((Lx_array_len, kT_array_len))
    for Lx_idx in range(Lx_array_len):
        with objmode(start='f8'):
            start = time.time()

        Lx = Lx_array[Lx_idx]
        area = Lx * Ly
        states_len = 2**area
        right_neighbors, bottom_neighbors = precompute_neighbors(Lx, Ly)
        half_states = states_len // 2
        energies = np.empty(half_states)

        for state in prange(half_states):
            E = 0.0
            for i in range(area):
                spin_i = 1 if (state >> i) & 1 else -1

                x_i = i // Ly
                y_i = i % Ly
                
                x_right, y_right = right_neighbors[x_i, y_i]
                idx_right = x_right * Ly + y_right
                spin_right = 1 if (state >> idx_right) & 1 else -1
                E -= J * spin_i * spin_right
                
                x_bottom, y_bottom = bottom_neighbors[x_i, y_i]
                idx_bottom = x_bottom * Ly + y_bottom
                spin_bottom = 1 if (state >> idx_bottom) & 1 else -1
                E -= J * spin_i * spin_bottom
                
            energies[state] = E
        
        for kT_idx in prange(kT_array_len):
            kT = kT_array[kT_idx]

            log_probs = -energies / kT
            max_log_prob = np.max(log_probs)
            probabilities = 2 * np.exp(log_probs - max_log_prob)
            
            Z = np.sum(probabilities)
            normed_mean_energies[Lx_idx, kT_idx] = np.dot(energies, probabilities) / Z / area

        with objmode():
            print("Lx =", Lx, "done in", round(time.time() - start, 4), "seconds")

    return normed_mean_energies

In [ ]:
Lx_array = np.arange(2, 16).astype(int)
kT_array = np.arange(1.0, 10.0, 0.2)
Ly = 2

start_time = time.time()
results = compute_normed_mean_energy(Lx_array, Ly, kT_array)
full_time = time.time() - start_time 
np.save("normed_mean_energy.npy", results)

Lx = 2 done in 0.0159 seconds
Lx = 3 done in 0.0115 seconds
Lx = 4 done in 0.0097 seconds
Lx = 5 done in 0.0095 seconds
Lx = 6 done in 0.0087 seconds
Lx = 7 done in 0.0093 seconds
Lx = 8 done in 0.0097 seconds
Lx = 9 done in 0.013 seconds
Lx = 10 done in 0.0304 seconds
Lx = 11 done in 0.0869 seconds
Lx = 12 done in 0.3251 seconds
Lx = 13 done in 1.267 seconds
Lx = 14 done in 5.0342 seconds


In [6]:
print(f"Full time spent: {full_time:.4f}")

Full time spent: 94.0482


In [7]:
saved_result = np.load("normed_mean_energy.npy")

In [10]:
saved_result[0]

array([-1.99598209, -1.98483772, -1.96114384, -1.92211527, -1.86785426,
       -1.80082536, -1.72475185, -1.6435674 , -1.56072701, -1.47890165,
       -1.39994653, -1.32501373, -1.2547137 , -1.18927202, -1.12865902,
       -1.07268721, -1.02108034, -0.97351979, -0.92967447, -0.88921917,
       -0.85184512, -0.81726544, -0.78521735, -0.75546244, -0.72778573,
       -0.70199418, -0.67791481, -0.65539287, -0.63428991, -0.61448208,
       -0.59585849, -0.57831975, -0.56177666, -0.54614905, -0.53136477,
       -0.51735872, -0.50407213, -0.4914518 , -0.4794495 , -0.46802143,
       -0.45712773, -0.44673208, -0.43680131, -0.4273051 , -0.41821565])